In [1]:
import pandas as pd
INTERIM = "/mnt/c/Users/r4gbi/Desktop/Proyecto_Residencias/data/interim"

def leer_fasta(path):
    """Lee un FASTA y devuelve {encabezado: secuencia}.

    El encabezado se recorta al PRIMER TOKEN (line[1:].split()[0]) a proposito:
    ARGNet, DIAMOND y Prodigal cortan los encabezados en el primer espacio, asi
    que para poder cruzar resultados despues hay que guardar la misma clave corta.
    """
    s = {}            # diccionario resultado
    h = None          # encabezado en curso
    b = []            # trozos de secuencia del encabezado en curso
    with open(path) as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith(">"):             # linea de encabezado
                if h: s[h] = "".join(b)          # cierra el registro anterior
                h = line[1:].split()[0]; b = []  # abre el nuevo, sin el ">"
            else:
                b.append(line.strip())           # acumula (el FASTA parte la secuencia en varias lineas)
        if h: s[h] = "".join(b)                  # cierra el ultimo registro
    return s

genes = leer_fasta(f"{INTERIM}/muestra_genes_completos.fasta")

READ = 150        # longitud de lectura, tope del rango documentado para argnet-s
n_reads = 0

with open(f"{INTERIM}/muestra_reads150.fasta", "w") as out:
    for gid, seq in genes.items():
        seq = seq.upper()                        # normaliza a mayusculas
        for i in range(len(seq) // READ):        # division entera -> solo ventanas completas
            inicio = i * READ
            fragmento = seq[inicio : inicio + READ]
            # el sufijo __r{i} permite reagrupar las lecturas por gen mas adelante
            out.write(f">{gid}__r{i}\n{fragmento}\n")
            n_reads += 1

print(f"Lecturas de 150 nt generadas: {n_reads}")
print(f"Genes de entrada: {len(genes)}")
print(f"Genes que no produjeron ninguna lectura (miden < 150 nt): "
      f"{sum(1 for s in genes.values() if len(s) < READ)}")

Lecturas de 150 nt generadas: 11392
Genes de entrada: 2000
Genes que no produjeron ninguna lectura (miden < 150 nt): 23


In [2]:
import pandas as pd, numpy as np
from sklearn.metrics import classification_report, confusion_matrix

ARGNET_OUT = "/home/r4gbi/ARGNet/results/argnet_reads150.txt"
LABELS = "/mnt/c/Users/r4gbi/Desktop/Proyecto_Residencias/data/interim/muestra_genes_labels.tsv"

# ---------- 1) Salida de ARGNet: una fila por LECTURA ----------
# sep=None + engine="python" -> pandas deduce el separador solo
reads = pd.read_csv(ARGNET_OUT, sep=None, engine="python")
print("Columnas crudas:", list(reads.columns))

# Se localiza la columna de prediccion POR CONTENIDO (la que trae ARG / non-ARG),
# no por posicion: asi el codigo no se rompe si cambiara el orden de columnas.
col_pred = next(c for c in reads.columns
                if reads[c].astype(str).str.strip().str.upper().isin(["ARG","NON-ARG"]).any())
# el identificador es la primera columna, salvo que esa sea justo la de prediccion
col_id   = reads.columns[0] if reads.columns[0] != col_pred else reads.columns[1]

# ARG -> 1, non-ARG -> 0
reads["pred_bin"] = (reads[col_pred].astype(str).str.strip().str.upper() == "ARG").astype(int)
# se quita el sufijo __r{i} para recuperar el gen del que vino cada lectura
reads["gid"] = reads[col_id].astype(str).str.split("__r").str[0]

print(f"Lecturas evaluadas: {len(reads)} | Genes representados: {reads['gid'].nunique()}")

# ---------- 2) Verdad de referencia, a nivel GEN ----------
truth = pd.read_csv(LABELS, sep="\t")
gid_col = truth.columns[0]                      # primera columna = identificador del gen
# se toma como etiqueta la primera columna cuyos valores sean todos 0 o 1
lab_col = next(c for c in truth.columns[1:]
               if pd.to_numeric(truth[c], errors="coerce").dropna().isin([0,1]).all())
truth = truth[[gid_col, lab_col]].rename(columns={gid_col:"gid", lab_col:"y_true"})
truth["y_true"] = truth["y_true"].astype(int)
print(f"Genes en referencia: {len(truth)} "
      f"(positivos={truth['y_true'].sum()}, negativos={(truth['y_true']==0).sum()})\n")

def reporte(y_true, y_pred, titulo):
    """Matriz de confusion + metricas por clase.

    En la matriz: filas = verdad, columnas = prediccion.
    [[VN, FP],
     [FN, VP]]   (clase 0 arriba, clase 1 abajo)
    """
    print(f"=== {titulo} ===")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred,
          target_names=["no-ARG (0)","ARG (1)"], digits=3))

# ---------- 3a) NIVEL LECTURA ----------
# merge interno: cada lectura recibe la etiqueta verdadera de su gen
rd = reads.merge(truth, on="gid", how="inner")
reporte(rd["y_true"], rd["pred_bin"], f"ARGNet-s | NIVEL LECTURA ({len(rd)} lecturas)")

# ---------- 3b) NIVEL GEN, regla OR ----------
# max() sobre pred_bin = 1 si al menos una lectura del gen salio ARG
g_or = reads.groupby("gid")["pred_bin"].max().rename("y_pred").reset_index()
d_or = truth.merge(g_or, on="gid", how="left")          # left: conserva los 2000 genes
print(f"Genes sin NINGUNA lectura en la salida: {d_or['y_pred'].isna().sum()}")
d_or["y_pred"] = d_or["y_pred"].fillna(0).astype(int)   # sin lecturas -> se asume no-ARG
reporte(d_or["y_true"], d_or["y_pred"], "ARGNet-s | NIVEL GEN - regla OR")

# ---------- 3c) NIVEL GEN, regla mayoria ----------
# mean() = fraccion de lecturas del gen que salieron ARG; >= 0.5 es mas de la mitad
g_maj = (reads.groupby("gid")["pred_bin"].mean() >= 0.5).astype(int).rename("y_pred").reset_index()
d_maj = truth.merge(g_maj, on="gid", how="left")
d_maj["y_pred"] = d_maj["y_pred"].fillna(0).astype(int)
reporte(d_maj["y_true"], d_maj["y_pred"], "ARGNet-s | NIVEL GEN - regla MAYORIA")

Columnas crudas: ['test_id', 'ARG_prediction', 'resistance_category', 'probability']
Lecturas evaluadas: 11392 | Genes representados: 1977
Genes en referencia: 2000 (positivos=1000, negativos=1000)

=== ARGNet-s | NIVEL LECTURA (11392 lecturas) ===
[[5894  117]
 [2724 2657]]
              precision    recall  f1-score   support

  no-ARG (0)      0.684     0.981     0.806      6011
     ARG (1)      0.958     0.494     0.652      5381

    accuracy                          0.751     11392
   macro avg      0.821     0.737     0.729     11392
weighted avg      0.813     0.751     0.733     11392

Genes sin NINGUNA lectura en la salida: 23
=== ARGNet-s | NIVEL GEN - regla OR ===
[[921  79]
 [411 589]]
              precision    recall  f1-score   support

  no-ARG (0)      0.691     0.921     0.790      1000
     ARG (1)      0.882     0.589     0.706      1000

    accuracy                          0.755      2000
   macro avg      0.787     0.755     0.748      2000
weighted avg      0

In [3]:
# ---------- 1) Vocabulario de categorias que ARGNet uso sobre nuestros datos ----------
print("=== Categorias que predice ARGNet ===")
print(reads.groupby("ARG_prediction")["resistance_category"].value_counts(), "\n")

# ---------- 2) Base de datos de origen de cada gen ----------
# drop_duplicates("gid") deja un renglon por gen en lugar de uno por lectura
g = reads.drop_duplicates("gid").copy()
# el primer token del encabezado, antes del primer "|", es la fuente
# (CARD, ResFinderFG, MEGARes...; los GCF_* son los negativos sacados de genomas)
g["fuente"] = g["gid"].str.split("|").str[0]
print("=== Genes por base de datos de origen ===")
print(g["fuente"].value_counts(), "\n")

# ---------- 3) Encabezados de ejemplo por fuente ----------
# sirve para disenar el extractor de clase verdadera: cada base codifica
# la clase de antibiotico de forma distinta dentro del encabezado
for f, sub in g.groupby("fuente"):
    print(f"--- {f}  (n={len(sub)}) ---")
    for h in sub["gid"].head(4):
        print("   ", h[:140])
    print()

=== Categorias que predice ARGNet ===
ARG_prediction  resistance_category
ARG             bacitracin             1157
                beta-lactam             731
                multidrug               361
                MLS                     266
                aminoglycoside           65
                tetracycline             55
                glycopeptide             53
                trimethoprim             40
                quinolone                10
                chloramphenicol          10
                mupirocin                 9
                peptide                   6
                qa_compound               4
                fosfomycin                4
                ethambutol                3
Name: count, dtype: int64 

=== Genes por base de datos de origen ===
fuente
ResFinderFG        475
CARD               328
GCF_000412675.1    209
GCF_000008865.2    175
GCF_000009045.1    175
GCF_000025985.1    169
GCF_000005845.2    153
MEGARes            112
GCF_0

In [4]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def fila(nombre, y_true, y_pred):
    """Devuelve (nombre, recall, precision, f1, accuracy) para la CLASE ARG unicamente.

    labels=[1] -> solo la clase positiva.  average=None -> no promedia con la clase 0.
    """
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=[1], average=None)
    return (nombre, r[0], p[0], f[0], accuracy_score(y_true, y_pred))

tabla = pd.DataFrame([
    fila("ARGNet-s  nivel lectura", rd["y_true"],    rd["pred_bin"]),
    fila("ARGNet-s  gen, regla OR", d_or["y_true"],  d_or["y_pred"]),
    fila("ARGNet-s  gen, mayoria",  d_maj["y_true"], d_maj["y_pred"]),
], columns=["metodo","recall_ARG","precision_ARG","f1_ARG","accuracy"]).round(3)

print(tabla.to_string(index=False))
print("\nReferencia argnet-l sobre genes completos: recall_ARG = 0.553")
print("Referencia del articulo (lecturas cortas de nucleotido): recall 0.919, precision 0.989")
print(f"\nGenes sin ninguna lectura: {2000 - reads['gid'].nunique()}")

                 metodo  recall_ARG  precision_ARG  f1_ARG  accuracy
ARGNet-s  nivel lectura       0.494          0.958   0.652     0.751
ARGNet-s  gen, regla OR       0.589          0.882   0.706     0.755
 ARGNet-s  gen, mayoria       0.475          0.983   0.641     0.734

Referencia argnet-l sobre genes completos: recall_ARG = 0.553
Referencia del articulo (lecturas cortas de nucleotido): recall 0.919, precision 0.989

Genes sin ninguna lectura: 23


In [5]:
# ---------- Recall por base de datos, solo sobre los POSITIVOS ----------
pos = truth[truth["y_true"] == 1].copy()
pos["fuente"] = pos["gid"].str.split("|").str[0]     # primer token = base de origen

# se usa la regla OR, la mas favorable a ARGNet
g_or_map = reads.groupby("gid")["pred_bin"].max()
pos["y_pred"] = pos["gid"].map(g_or_map).fillna(0).astype(int)

por_fuente = (pos.groupby("fuente")
                 .agg(genes=("y_true", "size"),        # cuantos positivos aporta la base
                      detectados=("y_pred", "sum")))   # cuantos de ellos detecto ARGNet
por_fuente["recall"] = (por_fuente["detectados"] / por_fuente["genes"]).round(3)
por_fuente["% del set positivo"] = (100 * por_fuente["genes"] / len(pos)).round(1)
por_fuente = por_fuente.sort_values("recall")          # peor detectada primero

print("=== Recall de ARGNet por base de datos de origen (solo positivos) ===")
print(por_fuente.to_string())

# ---------- Comprobacion: los genes sin lecturas, son positivos o negativos? ----------
# Un POSITIVO sin lecturas seria un falso negativo automatico y habria que descontarlo.
# Un NEGATIVO sin lecturas se cuenta como no-ARG, que es su etiqueta correcta: no distorsiona.
neg = truth[truth["y_true"] == 0]
sin_lecturas_pos = (~pos["gid"].isin(reads["gid"])).sum()
sin_lecturas_neg = (~neg["gid"].isin(reads["gid"])).sum()
print(f"\nGenes sin ninguna lectura -> positivos: {sin_lecturas_pos} | negativos: {sin_lecturas_neg}")

BASE = "/mnt/c/Users/r4gbi/Desktop/Proyecto_Residencias"
por_fuente.to_csv(f"{BASE}/results/recall_argnet_por_fuente.csv")
print(f"\nGuardado en {BASE}/results/recall_argnet_por_fuente.csv")

=== Recall de ARGNet por base de datos de origen (solo positivos) ===
               genes  detectados  recall  % del set positivo
fuente                                                      
ResFinderFG      475          97   0.204                47.5
AMRFinderPlus     49          35   0.714                 4.9
CARD             328         313   0.954                32.8
ResFinder         25          24   0.960                 2.5
MEGARes          112         109   0.973                11.2
ARGANNOT          11          11   1.000                 1.1

Genes sin ninguna lectura -> positivos: 0 | negativos: 23

Guardado en /mnt/c/Users/r4gbi/Desktop/Proyecto_Residencias/results/recall_argnet_por_fuente.csv


In [6]:
import pandas as pd, numpy as np, re

BASE       = "/mnt/c/Users/r4gbi/Desktop/Proyecto_Residencias"
ARGNET_OUT = "/home/r4gbi/ARGNet/results/argnet_reads150.txt"
LABELS     = f"{BASE}/data/interim/muestra_genes_labels.tsv"
PROB_MIN   = 0.8     # mismo umbral que usa el articulo de ARGNet para contar una clasificacion

# ============================================================================
# 1) EXTRACTOR DE CLASE VERDADERA
#    Cada base codifica la clase de antibiotico de forma distinta en el encabezado.
# ============================================================================

# Reglas por prefijo del nombre del gen. Se aplican cuando el encabezado no
# trae un campo de clase explicito.
REGLAS = [
    (r"^(bla|oxa|tem|shv|ctx|kpc|ndm|imp|vim|cmy|ges|per|veb|oxy|mox|fox|act|adc|cfx|pse|car|sfh|len|cep|bl2|pbp|mec|amp)", "beta-lactam"),
    (r"^(aac|aph|ant|aad|arm|rmt|npm|str|sat|spc)", "aminoglycoside"),
    (r"^tet", "tetracycline"),
    (r"^van", "glycopeptide"),
    (r"^(erm|mef|msr|mph|lnu|lin|lsa|vga|vat|cfr|ere)", "MLS"),
    (r"^(sul|fol)", "sulfonamide"),
    (r"^dfr", "trimethoprim"),
    (r"^(qnr|gyr|par|oqx|qep|nor)", "quinolone"),
    (r"^(cat|flo|cml|cmx|cmr)", "chloramphenicol"),
    (r"^(mcr|pmr|ept|arn|bas)", "peptide"),
    (r"^fos", "fosfomycin"),
    (r"^(arr|rph|rpo|rif|iri)", "rifampin"),
    (r"^(bac|bcr|upp)", "bacitracin"),
    (r"^(mup|ile)", "mupirocin"),
    (r"^(mex|acr|mdt|emr|tmex|mate|mep|sme|ttg|opr|tol|cme)", "multidrug"),
    (r"^(qac|sug|smr)", "qa_compound"),
    (r"^(emb|eth|inh|kat|pnc)", "ethambutol"),
]

# MEGARes trae la clase explicita en el campo 4 del encabezado
MEGARES = {"betalactams":"beta-lactam","aminoglycosides":"aminoglycoside","tetracyclines":"tetracycline",
 "glycopeptides":"glycopeptide","macrolides":"MLS","lincosamides":"MLS","streptogramins":"MLS","mls":"MLS",
 "multi-drug_resistance":"multidrug","phenicol":"chloramphenicol","sulfonamides":"sulfonamide",
 "trimethoprim":"trimethoprim","fluoroquinolones":"quinolone","fosfomycin":"fosfomycin",
 "bacitracin":"bacitracin","cationic_antimicrobial_peptides":"peptide","rifampin":"rifampin",
 "mupirocin":"mupirocin","aminocoumarins":"aminocoumarin","nitroimidazoles":"nitroimidazole",
 "metronidazole":"nitroimidazole","thiostrepton":"MLS","elfamycins":"elfamycin"}

# ARG-ANNOT trae la clase como prefijo entre parentesis: (Bla)blaSFH-1
ARGANNOT = {"bla":"beta-lactam","agly":"aminoglycoside","tet":"tetracycline","gly":"glycopeptide",
 "phe":"chloramphenicol","mls":"MLS","sul":"sulfonamide","tmt":"trimethoprim","flq":"quinolone",
 "col":"peptide","rif":"rifampin","fos":"fosfomycin","ntmdz":"nitroimidazole","tric":"triclosan"}

def _por_nombre(nombre):
    """Deduce la clase a partir del prefijo del nombre del gen. None si ninguna regla aplica."""
    n = nombre.strip().lower()
    for patron, clase in REGLAS:
        if re.match(patron, n):
            return clase
    return None

def clase_verdadera(gid):
    """Clase real del gen segun su encabezado. None = no se pudo determinar."""
    campos = gid.split("|")
    fuente = campos[0]

    if fuente == "MEGARes" and len(campos) >= 4:          # campo 4 = clase explicita
        return MEGARES.get(campos[3].strip().lower()) or _por_nombre(campos[-1])

    if fuente == "ARGANNOT":                              # prefijo (Bla), (Phe), (Gly)...
        m = re.search(r"\(([A-Za-z]+)\)", gid)
        if m and ARGANNOT.get(m.group(1).lower()):
            return ARGANNOT[m.group(1).lower()]
        m2 = re.search(r"\)([A-Za-z0-9_\-\(\)']+)", gid)
        return _por_nombre(m2.group(1)) if m2 else None

    if fuente == "CARD":                                  # ultimo campo = nombre del gen
        return _por_nombre(campos[-1])

    if fuente == "AMRFinderPlus" and len(campos) >= 7:    # campo 7 = simbolo del gen
        return _por_nombre(campos[6]) or _por_nombre(campos[5])

    if fuente == "ResFinder" and len(campos) >= 2:        # nombre antes del primer guion bajo
        return _por_nombre(campos[1].split("_")[0])

    if fuente == "ResFinderFG" and len(campos) >= 2:      # campo 2 = familia/mecanismo
        return _por_nombre(campos[1])

    return None

# ============================================================================
# 2) CARGA DE DATOS
# ============================================================================
reads = pd.read_csv(ARGNET_OUT, sep=None, engine="python")
reads["pred_bin"] = (reads["ARG_prediction"].astype(str).str.strip().str.upper() == "ARG").astype(int)
reads["gid"]      = reads["test_id"].astype(str).str.split("__r").str[0]
reads["probability"] = pd.to_numeric(reads["probability"], errors="coerce")

truth = pd.read_csv(LABELS, sep="\t")
gid_col = truth.columns[0]
lab_col = next(c for c in truth.columns[1:]
               if pd.to_numeric(truth[c], errors="coerce").dropna().isin([0,1]).all())
truth = truth[[gid_col, lab_col]].rename(columns={gid_col:"gid", lab_col:"y_true"})
truth["y_true"] = truth["y_true"].astype(int)

# ============================================================================
# 3) CLASE VERDADERA DE CADA POSITIVO + COBERTURA DEL EXTRACTOR
# ============================================================================
pos = truth[truth["y_true"] == 1].copy()
pos["fuente"] = pos["gid"].str.split("|").str[0]
pos["clase_real"] = pos["gid"].apply(clase_verdadera)

cob = pos["clase_real"].notna()
print(f"Cobertura del extractor: {cob.sum()}/{len(pos)} positivos con clase asignada "
      f"({100*cob.mean():.1f}%)")
print("\nSin clasificar, por base de datos:")
print(pos.loc[~cob, "fuente"].value_counts().to_string() or "  ninguno")
print("\nDistribucion de la clase REAL del set positivo:")
print(pos.loc[cob, "clase_real"].value_counts().to_string())

# ============================================================================
# 4) CATEGORIA PREDICHA A NIVEL GEN
#    Solo lecturas llamadas ARG y con probabilidad >= 0.8 (el umbral del paper).
#    La categoria del gen es la MODA de sus lecturas que pasan el filtro.
# ============================================================================
conf = reads[(reads["pred_bin"] == 1) & (reads["probability"] >= PROB_MIN)]
print(f"\nLecturas ARG con probabilidad >= {PROB_MIN}: {len(conf)} de "
      f"{int(reads['pred_bin'].sum())} llamadas ARG")

moda = (conf.groupby("gid")["resistance_category"]
            .agg(lambda s: s.value_counts().idxmax())
            .rename("clase_pred"))
pos["clase_pred"] = pos["gid"].map(moda)

# ============================================================================
# 5) CONCORDANCIA
# ============================================================================
ev = pos[pos["clase_real"].notna() & pos["clase_pred"].notna()].copy()
ev["acierto"] = (ev["clase_real"] == ev["clase_pred"])

print("\n" + "="*68)
print(f"Genes positivos evaluables (clase real conocida Y detectados con prob>={PROB_MIN}): {len(ev)}")
print(f"CONCORDANCIA DE CATEGORIA: {ev['acierto'].sum()}/{len(ev)} = {ev['acierto'].mean():.3f}")
print(f"Referencia del articulo (lecturas cortas de nucleotido): 0.911")
print("="*68)

print("\n--- Concordancia por clase real ---")
porclase = (ev.groupby("clase_real")
              .agg(genes=("acierto","size"), aciertos=("acierto","sum")))
porclase["concordancia"] = (porclase["aciertos"]/porclase["genes"]).round(3)
print(porclase.sort_values("concordancia").to_string())

print("\n--- A que categoria se van los errores (clase real -> clase predicha) ---")
err = ev[~ev["acierto"]]
print(err.groupby(["clase_real","clase_pred"]).size()
         .sort_values(ascending=False).head(15).to_string())

print("\n--- Comprobacion de la sospecha sobre bacitracina ---")
n_bac_real = (ev["clase_real"] == "bacitracin").sum()
n_bac_pred = (ev["clase_pred"] == "bacitracin").sum()
print(f"Genes cuya clase REAL es bacitracina:    {n_bac_real}")
print(f"Genes a los que ARGNet le ASIGNO bacitracina: {n_bac_pred}")
if n_bac_real > 0:
    print(f"Razon predicho/real: {n_bac_pred/max(n_bac_real,1):.1f}x")
print("Una razon muy por encima de 1 indica que la herramienta recurre a esa clase sin sustento.")

ev[["gid","fuente","clase_real","clase_pred","acierto"]].to_csv(
    f"{BASE}/results/concordancia_categoria_argnet.csv", index=False)
print(f"\nGuardado en {BASE}/results/concordancia_categoria_argnet.csv")

Cobertura del extractor: 763/1000 positivos con clase asignada (76.3%)

Sin clasificar, por base de datos:
fuente
CARD             132
ResFinderFG       93
AMRFinderPlus     10
MEGARes            1
ResFinder          1

Distribucion de la clase REAL del set positivo:
clase_real
glycopeptide       199
trimethoprim       199
beta-lactam        137
aminoglycoside      75
tetracycline        43
MLS                 29
multidrug           25
chloramphenicol     22
fosfomycin          12
quinolone            7
peptide              6
rifampin             4
qa_compound          3
nitroimidazole       1
bacitracin           1

Lecturas ARG con probabilidad >= 0.8: 1297 de 2774 llamadas ARG

Genes positivos evaluables (clase real conocida Y detectados con prob>=0.8): 351
CONCORDANCIA DE CATEGORIA: 25/351 = 0.071
Referencia del articulo (lecturas cortas de nucleotido): 0.911

--- Concordancia por clase real ---
                 genes  aciertos  concordancia
clase_real                              